# Cryptography project SW6

## Generalities

This project aims to implement two side channel attacks that recover components of the RSA private key from a known fragment. More specifically, we are going to consider both the textbook implementation of RSA and the CRT variant and try to recover the private exponent $d$ and the $d_p$ CRT coefficient respectively. We are going to assume that the RSA implementations in question use a small public exponent (we will consider $e = 65537$ as all real world RSA implementations use that exponent).

Our work will be based on "Survey: Recovering cryptographic keys from partial information, by example" by Micheli and Heninger ([DOI](https://doi.org/10.62056/ahjbksdja)), specifically chapters 4.2.7 and 4.2.9.

The algorithms implemented here will make use of lattices, lattice reduction algorithms and polynomial root finding algorithms. Mathematical functions and abstractions will be provided by SageMath.

The high level structure of the class of key-recovery attacks against RSA implemented here is the following:

1. Cast the problem as finding small roots of a polynomial
2. Construct a lattice and find the shortest vector of the rediced basis using LLL
3. Extract a polynomial from the shortest vector and find its roots

## Auxiliary classes

To easily generate problem instances, we define the following container classes `RSA` and `RSA_CRT` that contain a key-pair for respectively textbook RSA and RSA with the CRT optimization, both with a factory function to generate a new random instances. Each of the two containers also provide functions to leak parts of the private key parts.

In [1]:
# Generate two primes that can be used as the private primes in RSA
def generate_rsa_primes(bit_length, relative_prime):
    while True:
        p = random_prime(2**(bit_length // 2 - 1), lbound=2**(bit_length // 2 - 2), proof=false)
        q = random_prime(2**(bit_length // 2 - 1), lbound=2**(bit_length // 2 - 2), proof=false)

        # Check that they are correct
        gcd_p = gcd(p - 1, relative_prime)
        gcd_q = gcd(q - 1, relative_prime)
        if gcd_p == 1 and gcd_q == 1:
            return (p, q)


# Base container of a textbook RSA instance
class RSA:
    e = 65537 # Public exponent used in all real implementations
    
    def __init__(self, bit_length, n, p, q):
        self.bit_length = bit_length
        self.n = n
        self._p = p
        self._q = q
        self._d = self.e.inverse_mod((self._p - 1) * (self._q - 1))

    @staticmethod
    def _leak(val, bit_length, n_bits, bit_offset):
        if n_bits + bit_offset > bit_length:
            raise ValueError()
        mask = 2**n_bits - 1
        mask = mask << bit_offset
        mid_b = Integer(val) & mask
        return mid_b
        
    def leak_p_mid_bits(self, n_bits, bit_offset):
        return self._leak(self._p, self.bit_length//2, n_bits, bit_offset)
    
    def leak_p_msb(self, n_bits):
        return self.leak_p_mid_bits(n_bits, self.bit_length//2 - n_bits)
    
    def leak_p_lsb(self, n_bits):
        return self.leak_p_mid_bits(n_bits, 0)

    def leak_q_mid_bits(self, n_bits, bit_offset):
        return self._leak(self._q, self.bit_length//2, n_bits, bit_offset)
    
    def leak_q_msb(self, n_bits):
        return self.leak_q_mid_bits(n_bits, self.bit_length//2 - n_bits)
    
    def leak_q_lsb(self, n_bits):
        return self.leak_q_mid_bits(n_bits, 0)
    
    def leak_private_exp_mid_bits(self, n_bits, bit_offset):
        return self._leak(self._d, self.bit_length, n_bits, bit_offset)
    
    def leak_private_exp_msb(self, n_bits):
        return self.leak_private_exp_mid_bits(n_bits, self.bit_length - n_bits)
    
    def leak_private_exp_lsb(self, n_bits):
        return self.leak_private_exp_mid_bits(n_bits, 0)

    @classmethod
    def new_random(klass, bit_length):
        p, q = generate_rsa_primes(bit_length, klass.e)
        n = p * q
        return klass(bit_length, n, p, q)


# Base container of an RSA instance implemented using the CRT
class RSA_CRT(RSA):
    def __init__(self, bit_length, n, p, q):
        super().__init__(bit_length, n, p, q)
        self._dp = self._d.mod(self._p - 1)
        self._dq = self._d.mod(self._q - 1)

    def leak_dp_mid_bits(self, n_bits, bit_offset):
        return self._leak(self._dp, self.bit_length//2, n_bits, bit_offset)
    
    def leak_dp_msb(self, n_bits):
        return self.leak_dp_mid_bits(n_bits, self.bit_length//2 - n_bits)
    
    def leak_dp_lsb(self, n_bits):
        return self.leak_dp_mid_bits(n_bits, 0)
    
    def leak_dq_mid_bits(self, n_bits, bit_offset):
        return self._leak(self._dq, self.bit_length//2, n_bits, bit_offset)
    
    def leak_dq_msb(self, n_bits):
        return self.leak_dp_mid_bits(n_bits, self.bit_length//2 - n_bits)
    
    def leak_dq_lsb(self, n_bits):
        return self.leak_dp_mid_bits(n_bits, 0)

We also provide some other miscellaneous utilities used in the implemented functions.

In [2]:
class Result:
    def __init__ (self, v):
        self._val = v

    def unwrap(self):
        return self._val

    def __str__(self):
        return f"Result containing {self._val}"


class Err:
    def __init__ (self, e):
        self._err = e
    
    def unwrap(self):
        raise RuntimeError(self._err)

    def __str__(self):
        return f"Error containing '{self._err}'"


def report_execution_time(f, *args, **kwargs):
    import time
    s = time.time()
    r = f(*args, **kwargs)
    e = time.time()
    print(f'Elapsed: {e-s}s')
    return r

## Recovery of $d_p$ from a large contiguous chunk

This is the first attack we are going to implement and consists of recovering the CRT coefficient $d_p$ with knowledge of a large continuous part of it. We will implement the attack in all three possible cases: knowledge of the most significant bits, knowledge of the least significant bits and knowledge of the middle bits. We will assume to only know the public part of the RSA key (i.e. $N$ and $e$) and no other information about anny part of the private key apart from the leaked bits.

### Known most significant bits

We implemented this case exactly as the reference paper outlined in section 4.2.7 with the `reconstruct_dp_lsb` function, which itself is based upon the lattice algorithm for recovering $p$ described in section 4.2.2, implemented in `try_extract_lsb_with_lattice`. This implementation will work for $2^l < p^{1/3} = N^{1/6}$, with $l$ being the number of unknown bits. By increasing the dimension of the lattice, this method can be pushed to work up to $2^l < p^{1/2} = N^{1/4}$.

In [3]:
def try_extract_lsb_with_lattice(rsa, A, l):
    # Check that our construction works for the required number of
    # unknown bits: our reconstructed part r should be less than
    # p^(1/3), which is roughly n^(1/6)
    assert l < rsa.bit_length // 6
    
    R = 2**l
    P.<X> = PolynomialRing(ZZ)
        
    # Construct the lattice and calculate its reduced basis
    B = matrix(ZZ, [[R*R,  R*A, 0    ],
                    [0,    R,   A    ],
                    [0,    0,   rsa.n]])
    B = B.LLL()

    # Extract the shortest vector
    v = B[0].list()

    # Compute the coefficients of the polynomial
    v[0] = v[0] >> l*2
    v[1] = v[1] >> l

    # Construct the corresponding polynomial f
    f = v[0] * (X**2) + v[1] * X + v[2]

    # Compute the roots of f and check if among the roots we find 
    # one that summed to A results in a factorization of N
    roots = f.roots()
    for root in roots:
        r = Integer(root[0])
        if gcd(A + r, rsa.n) != 1:
            return Result(r)
    return Err(f"Could not find solutions")


def reconstruct_dp_lsb(rsa, a, l, premul=1, first_kp=1, last_kp=RSA.e):
    inv_e = rsa.e.inverse_mod(rsa.n)
    for kp in range(first_kp, last_kp):
        # Calculate A
        A = a + inv_e * (kp - 1)

        # Multiply A by a given factor, this will be used later on
        # to implement reconstruction from known LSB
        A *= premul

        # Use the lattice algorithm to try to find a solution
        # If we have found a solution, we are successful and we end the computation
        res = try_extract_lsb_with_lattice(rsa, A, l)
        if type(res) == Result:
            # Check for spurious solutions
            r = res.unwrap()
            d_p = A + r * premul.inverse_mod(rsa.n)
            p = gcd(d_p, rsa.n)
            if gcd(p, rsa.n) == p:
                return Result((kp, r))
    return Err(f"Could not find solutions in the [{first_kp}:{last_kp}) range")

Since we need to try the computation for each $1 \leq k_p \leq \mathrm{RSA.e} = 65537$, we can easily speed up the execution by introducing some coarse-grain parallelism. We create a pool of worker processes (not threads to side-step python's GIL), each executing the algorithm in non-overlapping ranges; the "main" process acts as an orchestrator and keeps track of the results found so far. To allow for reusing this schduling mechanisms also with the other algorithms, we extracted the scheduling logic into a simple function, `schedule_block`.

`reconstruct_dp_lsb_mp` executes `reconstruct_dp_lsb` using this parallelization scheme and it allows us to reduce the execution time from ~6s to ~1.5s (6x speedup). Everything has been executed on a Linux laptop with 16Gb of RAM and a Ryzen 5700U, plugged into the power outlet.

In [4]:
def schedule_block(start, end, step, invocation, procs=None):
    import concurrent.futures as cc
    import os
    if procs is None:
        procs = os.cpu_count()
    with cc.ProcessPoolExecutor(max_workers=procs) as pool:
        for k in range(start, end, procs*step):
            tasks = set()
            for i in range(0, procs):
                start_k = min(end, i*step + k)
                last_k  = min(end, (i+1)*step + k - 1)
                f = pool.submit(*invocation, start_k, last_k)
                tasks.add(f)
            done, _ = cc.wait(tasks, return_when=cc.ALL_COMPLETED)
            for f in done:
                if type(f.result()) != Err:
                    return f.result()
    return Err("No scheduled block solved the problem")


def reconstruct_dp_lsb_mp(rsa, a, l, premul=1, granule=500):
    return schedule_block(1, rsa.e, granule, (reconstruct_dp_lsb, rsa, a, l, premul))

First, we will test our implementation with the example values used in the paper.

In [5]:
bit_length = 240
l = 30 # number of unknown bits

rsa = RSA_CRT(bit_length,
              0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5,
              0x68323401cb3a10959e7bfdc0873209,
              0xbd61131f8df72405f97bb4b961528d)
a = rsa.leak_dp_msb(bit_length//2 - l)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} LSBs')

kp, r = report_execution_time(reconstruct_dp_lsb_mp, rsa, a, l).unwrap()
r_dp = a + r
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5
We have leaked 0x25822d06984a06be5596fcc0000000 and want to reconstruct the 30 LSBs
Elapsed: 1.8276028633117676s
We started with the following private exponent: 0x25822d06984a06be5596fcf9d9b141
We reconstructed using kp = 23592 the following private exponent: 0x25822d06984a06be5596fcf9d9b141
Have we succeeded? True


Then, we will test it with a random instance of RSA w/ CRT.

In [6]:
rsa = RSA_CRT.new_random(bit_length)
a = rsa.leak_dp_msb(bit_length//2 - l)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} LSBs')

kp, r = report_execution_time(reconstruct_dp_lsb_mp, rsa, a, l).unwrap()
r_dp = a + r
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x1cfbfea85fedb55b1c250bfac77ee4064cec998a0ca2ea4273cc44f9037f
We have leaked 0x4de101fa67cc88e5af8e9d00000000 and want to reconstruct the 30 LSBs
Elapsed: 3.1615259647369385s
We started with the following private exponent: 0x4de101fa67cc88e5af8e9d3e246311
We reconstructed using kp = 44220 the following private exponent: 0x4de101fa67cc88e5af8e9d3e246311
Have we succeeded? True


### Known least significant bits

Applying the same reasoning outlined in 4.2.7 but using $d_p = 2^m r + a$ where $a$ are the $m$ leaked LSBs and $r$ are the unknown bits, we get that the polynomial will have the form $f(x) = 2^m x + A$. By multiplying everything by $2^{-m} \mod N$, we can obtain the same polynomial as the one used in the previous section and reuse the same algorithm. Finally, we will need to multiply $r$ by $2^m$ to obtain the correct value. The reconstructed $d_p$ will be $a + r$ as before.

Since we are reusing our previous routine, which worked with $2^l < N^{1/6}$ ($l$ being the number of unknown LSBs), this method will work if we know at least $m > \mathrm{num\_bits}(N) / 6$ LSBs of $d_p$.

In [7]:
def reconstruct_dp_msb_mp(rsa, a, m):
    # Since our reconstruct_dp_msb() works to recover N^1/6 bits
    # of dp, this method will work when at least N^1/6 bits of dp are known.
    assert m > rsa.bit_length // 6

    l = rsa.bit_length//2 - m
    R = 2**m
    mul = R.inverse_mod(rsa.n)
    kp, r = reconstruct_dp_lsb_mp(rsa, a, l, premul=mul).unwrap()
    return Result((kp, r * R))

As before, we first test with the example values used in the paper...

In [8]:
bit_length = 240
l = 30                # number of unknown dp MSBs
m = bit_length//2 - l # number of known dp LSBs

### Values in the paper
rsa = RSA_CRT(bit_length,
              0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5,
              0x68323401cb3a10959e7bfdc0873209,
              0xbd61131f8df72405f97bb4b961528d)
a = rsa.leak_dp_lsb(m)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} MSBs')

kp, r = report_execution_time(reconstruct_dp_msb_mp, rsa, a, m).unwrap()
r_dp = r + a
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5
We have leaked 0x2984a06be5596fcf9d9b141 and want to reconstruct the 30 MSBs
Elapsed: 1.6958808898925781s
We started with the following private exponent: 0x25822d06984a06be5596fcf9d9b141
We reconstructed using kp = 23592 the following private exponent: 0x25822d06984a06be5596fcf9d9b141
Have we succeeded? True


... then with a random instance.

In [9]:
### Random instance
rsa = RSA_CRT.new_random(bit_length)
a = rsa.leak_dp_lsb(m)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} MSBs')

kp, r = report_execution_time(reconstruct_dp_msb_mp, rsa, a, m).unwrap()
r_dp = r + a
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x3c836c79f26070349450ba0834c2266fcb5fa2d0d2d04cd05cacac514adb
We have leaked 0xa26b1ba857af155d26c959 and want to reconstruct the 30 MSBs
Elapsed: 3.189455032348633s
We started with the following private exponent: 0x633eeb50a26b1ba857af155d26c959
We reconstructed using kp = 51962 the following private exponent: 0x633eeb50a26b1ba857af155d26c959
Have we succeeded? True


### Known middle bits

To handle this case, we can use the same reasoning outlined in section 4.2.7 but, instead, use the lattice algorithm to recover $p$ outlined in section 4.2.4. As explained in the paper, this approach works for $R^{20} < N$, with $R = 2^{\mathit{exp}},\,\mathit{exp} = \max\{l, m\}$ where $l,m$ are the bit lengths of the unknown lower and upper parts respectively.

The function `try_extract_mid_with_lattice` implements the algorithm exactly as described in section 4.2.4. `reconstruct_dp_mid`, on the other hand, adapts the previous algorithms for recovering $d_p$, applying the same reasoning as in 4.2.7.

In [10]:
def try_extract_mid_with_lattice(rsa, A, t, exp):
    N = rsa.n
    R = 2**exp
    R2 = R^2
    R3 = R^3
    assert exp*20 < rsa.bit_length
    
    P.<X,Y> = PolynomialRing(ZZ)

    # Construct the lattice and calculate its reduced basis
    B = matrix(ZZ,[[R3, 3*2^t*R3, 3*2^(2*t)*R3, 2^(3*t)*R3, 3*A*R2, 6*2^t*A*R2, 3*2^(2*t)*A*R2, 3*A^2*R, 3*2^t*A^2*R, A^3],
                   [0,  R3,       2*2^t*R3,     2^(2*t)*R3, 0,      2*A*R2,     2*2^t*A*R2,     0,       A^2*R,       0  ],
                   [0,  0,        R3,           2^t*R3,     0,      0,          A*R2,           0,       0,           0  ],
                   [0,  0,        0,            R3*N,       0,      0,          0,              0,       0,           0  ],
                   [0,  0,        0,            0,          R2,     2*2^t*R2,   2^(2*t)*R2,     2*A*R,   2*2^t*A*R,   A^2],
                   [0,  0,        0,            0,          0,      R2,         2^t*R2,         0,       A*R,         0  ],
                   [0,  0,        0,            0,          0,      0,          R2*N,           0,       0,           0  ],
                   [0,  0,        0,            0,          0,      0,          0,              R,       2^t*R,       A  ],
                   [0,  0,        0,            0,          0,      0,          0,              0,       R*N,         0  ],
                   [0,  0,        0,            0,          0,      0,          0,              0,       0,           N  ]])
    B = B.LLL()

    # Reconstruct bivariate polynomials for each row
    monomials = vector((X^3, X^2*Y, X*Y^2, Y^3, X^2, X*Y, Y^2, X, Y, 1))
    polys = []
    for r in B:
        r[0:4] /= R3
        r[4:7] /= R2
        r[7:9] /= R
        polys.append(r * monomials)

    # Construct the ideal over P[ZZ] with basis a set of the shortest vectors
    # then calculate the Groebner basis.
    #
    # We need to try increasing subsets since we need one where the polys
    # are not algebraically dependent. To check for algebraic independence,
    # avoid using the algebraic_dependence() function since if the polynomial
    # sequence is not dependent, it will compute seemingly indefinetly and 
    # eat all our RAM. Instead, check that the last poly in the Groebner base
    # has degree equal to 1, solve that to obtain the value of Y and substitute
    # it into the second to get X.
    #
    # I am not 100% sure this is mathematically guarateed or correct since this
    # math goes a bit over my head, but it works well enough, I guess.
    for i in range(2, len(polys)+1):
        # NOTE:
        # Working with mutlivariate polynomials slowly leaks memory.
        # It doesn't leak much, howver this functions will run 65 thousands times,
        # Quickly exausting the 16 GB of RAM of my setup. A workaround will be 
        # needed.
        #
        # See also: https://github.com/sagemath/sage/issues/32604
        gb = ideal(polys[0:i]).groebner_basis()
        if gb[-1].degree() != 1:
            continue

        y_roots = gb[-1].univariate_polynomial().roots()
        if len(y_roots) != 1:
            return Err("Could not solve the polynomial system")
        rm = ZZ(y_roots[0][0])

        px = gb[-2].substitute({Y: rm})
        x_roots = px.univariate_polynomial().roots()
        if len(x_roots) != 1:
            return Err("Could not solve the polynomial system")
        rl = ZZ(x_roots[0][0])
        
        return Result((rm, rl))
    return Err("Could not solve the polynomial system")
            

def reconstruct_dp_mid(rsa, a, l, m, t, first_kp=1, last_kp=RSA.e):
    exp = max(l, t)
    assert exp*20 < rsa.bit_length
    
    t_shift = l + m
    inv_e = rsa.e.inverse_mod(rsa.n)
    for kp in range(first_kp, last_kp):
        # Calculate A
        A = ZZ(mod(a + inv_e * (kp - 1), rsa.n))

        # Use the lattice algorithm to try to find a solution
        # If we have found a solution, we are successful and we end
        # the computation
        res = try_extract_mid_with_lattice(rsa, A, t_shift, exp)
        if type(res) == Result:
            rm, rl = res.unwrap()
            return Result((kp, rm, rl))
    return Err(f"Could not find solutions in the [{first_kp}:{last_kp}) range")

Unfortunately, it seems that Sage has a memory leak when evaluating multivariate polynomials over the integers (see for example [this issue](https://github.com/sagemath/sage/issues/32604)). Since we computing at most 10 groebner bases for each `try_extract_mid_with_lattice`, which itself is called at most 65 thousand times by `reconstruct_dp_mid`, we quickly exhaust 16GB of RAM (even for small mmoduli). To get around this, define a modified `schedule_block`, `schedule_block_leaky`, that closes the `ProcessPool` when each battery of workers finishes their tasks, allowing the OS to reclaim the leaked memory and keep the leak at bay.

`reconstruct_dp_mid_mp` uses this "anti-leak" scheduling function to implement a parallelized version of `reconstruct_dp_mid`. Execution times for this are worse than for the other two cases because:

1. We are doing many more computations (bigger matrix and many polynomial calculations);
2. We have some overhead from our leak-containment solution.

This translates into ~4 minutes of execution time instead of ~1 second as the other cases for the same modulus bit length.

In [11]:
# NOTE:
# Since we are slowly leaking memory when solving polynomials, define a
# modified version of the schedule_block function that terminates
# the ProcessPool every time the workers finish a set of kps, this way
# any unclaimed memory is reclaimed by the OS.
def schedule_block_leaky(start, end, step, invocation, procs=None):
    import concurrent.futures as cc
    import os
    if procs == None:
        procs = os.cpu_count()
    for k in range(start, end, procs*step):
        with cc.ProcessPoolExecutor(max_workers=procs) as pool:
            tasks = set()
            for i in range(0, procs):
                start_k = min(end, i*step + k)
                last_k  = min(end, (i+1)*step + k - 1)
                f = pool.submit(*invocation, start_k, last_k)
                tasks.add(f)
            done, _ = cc.wait(tasks, return_when=cc.ALL_COMPLETED)
            for f in done:
                if type(f.result()) != Err:
                    return f.result()
    return Err("Could not find a solution to the problem")


def reconstruct_dp_mid_mp(rsa, a, l, m, t, granule=40):
    return schedule_block_leaky(1, rsa.e, granule, (reconstruct_dp_mid, rsa, a, l, m, t))

As before, we can test with the values in the paper...

In [12]:
bit_length = 240
l = t = 8                 # number of unknown dp LSB and MSB respectively
m = bit_length//2 - (l+t) # number of known dp mid bits

### Values in the paper
rsa = RSA_CRT(bit_length,
              0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5,
              0x68323401cb3a10959e7bfdc0873209,
              0xbd61131f8df72405f97bb4b961528d)
a = rsa.leak_dp_mid_bits(m, l)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} LSBs and {t} MSBs')

kp, rm, rl = report_execution_time(reconstruct_dp_mid_mp, rsa, a, l, m, t).unwrap()
r_dp = 2**(l + m)*rm + a + rl
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x4d14933399708b4a5276373cb5b756f312f023c43d60b323ba24cee670f5
We have leaked 0x822d06984a06be5596fcf9d9b100 and want to reconstruct the 8 LSBs and 8 MSBs
Elapsed: 258.31086897850037s
We started with the following private exponent: 0x25822d06984a06be5596fcf9d9b141
We reconstructed using kp = 23592 the following private exponent: 0x25822d06984a06be5596fcf9d9b141
Have we succeeded? True


... and a random example.

In [13]:
bit_length = 240
l = t = 8                 # number of unknown dp LSB and MSB respectively
m = bit_length//2 - (l+t) # number of known dp mid bits

### Values in the paper
rsa = RSA_CRT.new_random(bit_length)
a = rsa.leak_dp_mid_bits(m, l)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(a)} and want to reconstruct the {l} LSBs and {t} MSBs')

kp, rm, rl = report_execution_time(reconstruct_dp_mid_mp, rsa, a, l, m, t).unwrap()
r_dp = 2**(l + m)*rm + a + rl
print(f'We started with the following private exponent: {hex(rsa._dp)}')
print(f'We reconstructed using kp = {kp} the following private exponent: {hex(r_dp)}')
print(f'Have we succeeded? {rsa._dp == r_dp}')

We have a 240-bit RSA modulus: 0x2e527532526b20367d8e003ab19b28302d426c448e4c077f997c837eddeb
We have leaked 0xe4f11d9be0bd173f86bd19a27000 and want to reconstruct the 8 LSBs and 8 MSBs
Elapsed: 182.71124815940857s
We started with the following private exponent: 0x1be4f11d9be0bd173f86bd19a2701d
We reconstructed using kp = 16142 the following private exponent: 0x1be4f11d9be0bd173f86bd19a2701d
Have we succeeded? True


## Recovery of $p$ from knowledge of the least significant bits of $d$

In this section we are going to implement our second attack: recovering one of the primes $p$ from knowledge of the LSBs of $d$ of a textbook RSA private key. This algorithm is described in section 4.2.9 of the reference paper.

**N.B.**: In this context, when we are referring to $p$ we are talking about _any_ of the two primes since for our algorithms they are indistinguishable and also are both equally viable in breaking RSA. So in the snippets below, when checking our work, we will be checking if we found $p$ _or_ $q$.

First, we are going to translate our knowledge into possible candidates of the LSBs of $p$, then we are going to reconstruct the MSBs using our previous `try_extract_lsb_with_lattice` algorithm until we can factor the public modulus. The snippet below implements the algorithm as outlined in section 4.2.9. Since at the limit the recovery methods for $p$ recover at least $N^{1/4}$, this method works when at least $N^{1/4}$ bits of $d$ are known. Our implementation, due to using smaller lattices, works with at least $N^{1/6}$ bits.

In [14]:
def reconstruct_p(rsa, d0, t, start_k=2, end_k=RSA.e):
    # Since our try_extract_lsb_with_lattice() works to recover N^1/6 bits
    # of p, this method will work when at least N^1/6 bits of d are known.
    assert t > rsa.bit_length // 6

    l = rsa.bit_length//2 - t  # Unknown bits of p
    R = 2**t
    mul = R.inverse_mod(rsa.n)
    
    S = var('S')
    P = var('P')

    # NOTE:
    # Unfortunately, it seeems that Sage 10.4 solve_mod function has a
    # memory leak in, most likely, the wrapped C code (specifically the leaky
    # line is https://github.com/sagemath/sage/blob/develop/src/sage/symbolic/relation.py#L1693).
    # See also https://github.com/sagemath/sage/issues/25515.
    for k in range(start_k, end_k):
        S_sols = solve_mod(rsa.e*d0 == 1 + k*(rsa.n - S + 1), R)
        for s in S_sols:
            s = ZZ(s[0])
            P_sols = solve_mod(P**2 - s * P + rsa.n == 0, R)
            for a in P_sols:
                a = ZZ(a[0])
                A = mul * a
                res = try_extract_lsb_with_lattice(rsa, A, l)
                if type(res) == Result:
                    r = gcd(a + res.unwrap() * R, rsa.n)
                    return Result(r)
    return Err(f"Could not find solutions in the [{start_k}:{end_k}) range")

Unfortunately, Sage has again a memory leak (this time in `solve_mod` since that too seems to be using polynomials internally, see comment in the code above for more details) so we are going to have to reuse our workaround to keep the RAM from filling.

In [15]:
# NOTE:
# Unfortunately, since we are leaking memory, to be able to successfully
# run the proof of concept code, we need to work around this in some way.
# Use the previously defined modified version of schedule_block function
# to try and mitigate the problem
def reconstruct_p_mp(rsa, d0, t, granule=500):
    return schedule_block_leaky(2, rsa.e, granule, (reconstruct_p, rsa, d0, t))

Now let us test this implementation on a known key-pair. We are going to use a key-pair with a modulus of 64-bits, since this method is much more computationally intensive than the previous methods (~3 minutes for 64-bit modulus instead of ~1 second for a 240-bit one like before) since we need to solve multiple modular equations, linear and quadratic, and we have have a permormance penalty due to the memory leak workaround.

In [16]:
bit_length = 64
l = 6                 # number of unknown p bits
t = bit_length//2 - l # number of known d bits

rsa = RSA(bit_length,
          0x2a9665805f8741d5,
          0x71cbf057,
          0x5fce53b3)
d0 = rsa.leak_private_exp_lsb(t)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(d0)} of d and want to reconstruct one of the primes')

r_p = report_execution_time(reconstruct_p_mp, rsa, d0, t).unwrap()

print(f'We started with the following primes: p: {hex(rsa._p)} - q: {hex(rsa._q)}')
print(f'We have reconstructed the following the prime: {hex(r_p)}')
print(f'Is it equal to one of the private primes? {rsa._p == r_p or rsa._q == r_p}')
print(f'Does it factor N? {gcd(r_p, rsa.n) != 1}')

We have a 64-bit RSA modulus: 0x2a9665805f8741d5
We have leaked 0x2bbb069 of d and want to reconstruct one of the primes
Elapsed: 191.76576828956604s
We started with the following primes: p: 0x71cbf057 - q: 0x5fce53b3
We have reconstructed the following the prime: 0x71cbf057
Is it equal to one of the private primes? True
Does it factor N? True


Now to cross-check, let us try a randomly generated key-pair with modulus of 64-bits, like before.

In [17]:
bit_length = 64
l = 6                 # number of unknown p bits
t = bit_length//2 - l # number of known d bits

rsa = RSA.new_random(bit_length)
d0 = rsa.leak_private_exp_lsb(t)

print(f'We have a {bit_length}-bit RSA modulus: {hex(rsa.n)}')
print(f'We have leaked {hex(d0)} of d and want to reconstruct one of the primes')

r_p = report_execution_time(reconstruct_p_mp, rsa, d0, t).unwrap()

print(f'We started with the following primes: p: {hex(rsa._p)} - q: {hex(rsa._q)}')
print(f'We have reconstructed the following the prime: {hex(r_p)}')
print(f'Is it equal to one of the private primes? {rsa._p == r_p or rsa._q == r_p}')
print(f'Does it factor N? {gcd(r_p, rsa.n) != 1}')

We have a 64-bit RSA modulus: 0x293f2fcb798225e5
We have leaked 0x1ad55a5 of d and want to reconstruct one of the primes
Elapsed: 40.203208923339844s
We started with the following primes: p: 0x7122a3f7 - q: 0x5d551603
We have reconstructed the following the prime: 0x7122a3f7
Is it equal to one of the private primes? True
Does it factor N? True


## Conclusions

We sucessfully defined PoC implementations for the algorithms outlined in sections 4.2.7 (for LSB, MSB and mid-bits cases) and 4.2.9 of our reference paper, showing that these algorithms are more than usable in a real-world scenario if optimised and improved to reach their theoretical limits.